# SP500 Executive Classification — Holdout Evaluation

Loads the fine-tuned model from HuggingFace and evaluates on 2,010 holdout examples.

**Runtime**: Any GPU (T4 is sufficient). A100 is faster.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load fine-tuned model from HuggingFace (~3-4 min)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "daresearch/sp500-exec-classifier",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("Model loaded and ready for inference.")

In [ ]:
# Clone repo for holdout data
import os
REPO_DIR = "/content/finetune_sp500"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daalbert981/finetune_sp500.git {REPO_DIR}

import csv, re, time, json
import numpy as np

# Load holdout data
HOLDOUT_PATH = os.path.join(REPO_DIR, "Training Datasets", "holdout_labeling_09072024_template.csv")
with open(HOLDOUT_PATH, "r", encoding="latin-1") as f:
    holdout_rows = list(csv.DictReader(f))

# Load system prompt from training data
DATA_PATH = os.path.join(REPO_DIR, "Training Datasets", "full_data_n_4977.jsonl")
with open(DATA_PATH, "r") as f:
    FULL_SYSTEM_PROMPT = json.loads(f.readline())["messages"][0]["content"]

RANK_LABELS = ["vp", "svp", "evp", "sevp", "dir", "sdir", "md", "smd", "se", "vc", "svc", "president"]
ROLE_LABELS = ["board", "ceo", "cxo", "primary", "support", "bu"]
ALL_LABELS = RANK_LABELS + ROLE_LABELS

print(f"Holdout examples: {len(holdout_rows)}")
print(f"System prompt: {len(FULL_SYSTEM_PROMPT)} chars")

In [ ]:
def build_input(row):
    """Build tokenized input for a single holdout row."""
    user_msg = (
        f"In {row['year']} the company '{row['company']}' had an executive "
        f"with the name {row['full_name']}, whose official role title was: '{row['role']}'."
    )
    messages = [
        {"role": "system", "content": FULL_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
        return_dict = True,
        reasoning_effort = "low",  # skip deep reasoning — classification doesn't need it
    )


def parse_output(text):
    """Parse <rank>vp=0;...</rank><role>board=0;...</role> into dict."""
    result = {}
    for tag in ["rank", "role"]:
        match = re.search(f"<{tag}>(.*?)</{tag}>", text)
        if match:
            for pair in match.group(1).split(";"):
                if "=" in pair:
                    k, v = pair.split("=", 1)
                    try: result[k.strip()] = int(v.strip())
                    except ValueError: result[k.strip()] = -1
    return result


# Quick sanity check — single prediction
test_input = build_input(holdout_rows[0])
with torch.no_grad():
    out = model.generate(**test_input.to("cuda"), max_new_tokens=100, do_sample=False)
decoded = tokenizer.decode(out[0][test_input["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print(f"Test: {holdout_rows[0]['full_name']} — {holdout_rows[0]['role']}")
print(f"Output: {decoded}")
print(f"Parsed: {parse_output(decoded)}")

In [ ]:
# Run predictions on all holdout examples
# Uses single-example inference to avoid Unsloth attention mask bug with batching

predictions = []
parse_failures = 0
t0 = time.time()

for i, row in enumerate(holdout_rows):
    inputs = build_input(row).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    parsed = parse_output(decoded)

    if not all(lbl in parsed for lbl in ALL_LABELS):
        parse_failures += 1

    predictions.append({
        "idx": i, "uniqueid": row["uniqueid"], "full_name": row["full_name"],
        "company": row["company"], "role": row["role"], "year": row["year"],
        "raw_output": decoded, "parsed": parsed,
    })

    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(holdout_rows) - i - 1) / rate
        print(f"  [{i+1}/{len(holdout_rows)}] {rate:.1f} ex/s, ETA {eta/60:.1f} min, parse failures: {parse_failures}")

elapsed = time.time() - t0
print(f"\nDone! {len(predictions)} predictions in {elapsed/60:.1f} min ({len(predictions)/elapsed:.1f} ex/s)")
print(f"Parse failures: {parse_failures}/{len(predictions)}")

## Results

In [ ]:
y_true = {lbl: [] for lbl in ALL_LABELS}
y_pred = {lbl: [] for lbl in ALL_LABELS}

for pred, row in zip(predictions, holdout_rows):
    for lbl in ALL_LABELS:
        y_true[lbl].append(int(row[lbl]))
        y_pred[lbl].append(pred["parsed"].get(lbl, -1))

print(f"{'Label':>10} | {'Acc':>6} | {'Prec':>6} | {'Recall':>6} | {'F1':>6} | {'Support':>7} | {'Parse%':>6}")
print("-" * 72)

label_metrics = {}
for lbl in ALL_LABELS:
    yt = np.array(y_true[lbl]); yp = np.array(y_pred[lbl])
    valid = (yp >= 0); parse_rate = valid.sum() / len(yp) * 100
    yt_v = yt[valid]; yp_v = yp[valid]
    if len(yt_v) == 0:
        label_metrics[lbl] = {"acc": 0, "prec": 0, "rec": 0, "f1": 0, "support": 0, "parse_rate": 0}
        print(f"{lbl:>10} | {'N/A':>6} | {'N/A':>6} | {'N/A':>6} | {'N/A':>6} | {0:>7} | {parse_rate:>5.1f}%")
        continue
    tp = ((yp_v==1)&(yt_v==1)).sum(); fp = ((yp_v==1)&(yt_v==0)).sum()
    fn = ((yp_v==0)&(yt_v==1)).sum(); tn = ((yp_v==0)&(yt_v==0)).sum()
    acc = (tp+tn)/len(yt_v)
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    support = int(yt_v.sum())
    label_metrics[lbl] = {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "support": support, "parse_rate": parse_rate}
    print(f"{lbl:>10} | {acc:>5.1%} | {prec:>5.1%} | {rec:>5.1%} | {f1:>5.1%} | {support:>7} | {parse_rate:>5.1f}%")

print("-" * 72)
avg_acc = np.mean([m["acc"] for m in label_metrics.values()])
avg_f1 = np.mean([m["f1"] for m in label_metrics.values()])
rank_f1 = np.mean([label_metrics[l]["f1"] for l in RANK_LABELS])
role_f1 = np.mean([label_metrics[l]["f1"] for l in ROLE_LABELS])
print(f"\nMacro Acc: {avg_acc:.1%}  |  Macro F1: {avg_f1:.1%}")
print(f"Rank F1:   {rank_f1:.1%}  |  Role F1:  {role_f1:.1%}")

In [ ]:
# Misclassified examples
misclassified = []
for pred, row in zip(predictions, holdout_rows):
    errors = {lbl: f"pred={pred['parsed'].get(lbl,-1)} truth={int(row[lbl])}" for lbl in ALL_LABELS if pred['parsed'].get(lbl,-1) != int(row[lbl])}
    if errors: misclassified.append((pred, row, errors))

print(f"Fully correct: {len(holdout_rows)-len(misclassified)}/{len(holdout_rows)} ({(len(holdout_rows)-len(misclassified))/len(holdout_rows):.1%})")
print(f"With errors: {len(misclassified)}/{len(holdout_rows)}\n")

for pred, row, errors in misclassified[:20]:
    print(f"{row['full_name']} - {row['role']} @ {row['company']} ({row['year']})")
    for lbl, err in errors.items(): print(f"    {lbl}: {err}")
    print()

In [ ]:
# Export full results to CSV
OUTPUT_CSV = "/content/holdout_results.csv"
with open(OUTPUT_CSV, "w", newline="") as f:
    fieldnames = ["uniqueid","year","company","full_name","role","raw_output"] + \
                 [f"truth_{l}" for l in ALL_LABELS] + [f"pred_{l}" for l in ALL_LABELS] + [f"correct_{l}" for l in ALL_LABELS]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for pred, row in zip(predictions, holdout_rows):
        out = {"uniqueid": row["uniqueid"], "year": row["year"], "company": row["company"],
               "full_name": row["full_name"], "role": row["role"], "raw_output": pred["raw_output"]}
        for lbl in ALL_LABELS:
            truth = int(row[lbl]); predicted = pred["parsed"].get(lbl, -1)
            out[f"truth_{lbl}"] = truth; out[f"pred_{lbl}"] = predicted; out[f"correct_{lbl}"] = int(predicted == truth)
        writer.writerow(out)

print(f"Results exported to {OUTPUT_CSV}")
try:
    from google.colab import files; files.download(OUTPUT_CSV)
except: pass